<a href="https://colab.research.google.com/github/Anusha-Antony/Agentic-AI-Mini-Project/blob/main/mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔐 CyberFeed — Endless Cybersecurity Fact Cards
An AI-powered scroll feed that generates live cybersecurity insights from Wikipedia using the Groq API.

**How it works:**
1. Wikipedia pages on the chosen topic are fetched and chunked into a knowledge base.
2. On each scroll/click, a random chunk is sent to an LLM which generates a fresh 1–2 sentence insight.
3. The card is displayed in a stylish Gradio interface.


In [6]:
# Cell 1 — Install dependencies
!pip install groq wikipedia gradio --quiet

In [7]:
# Cell 2 — Imports and API key setup
# ⚠️  NEVER hardcode your API key. Use Colab Secrets:
#   Runtime → Manage secrets → Add secret named GROQ_API_KEY

import os
import re
import random
import wikipedia
import gradio as gr
from groq import Groq

# Load API key from Colab Secrets
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('groq_key')
except Exception:
    # Fallback: read from environment variable
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')

if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_API_KEY not found. Add it via Runtime → Manage secrets.")

client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client initialised successfully.")

✅ Groq client initialised successfully.


In [8]:
!pip install wikipedia-api
import random
import wikipediaapi
import re


In [20]:
# Cell 3 — Knowledge base: Wikipedia fetch + chunking

# Global in-memory knowledge base (list of dicts with 'chunk' and 'source')
KNOWLEDGE_BASE = []

DEFAULT_PAGES = [
    "Computer security", "Cybercrime", "Malware", "Phishing",
    "Ransomware", "Firewall (computing)", "Encryption",
    "Data breach", "Social engineering (security)",
    "Denial-of-service attack"
]

wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="CyberFeed/1.0 (colab-project; educational use)"
)

def clean_text(text: str) -> str:
    text = re.sub(r"=+[^=]+=+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(text: str, chunk_size: int = 350) -> list:
    words = text.split()
    return [
        " ".join(words[i : i + chunk_size])
        for i in range(0, len(words), chunk_size)
        if words[i : i + chunk_size]
    ]

def load_knowledge_base(page_titles: list) -> str:
    global KNOWLEDGE_BASE
    KNOWLEDGE_BASE = []
    loaded, failed = [], []

    for title in page_titles:
        try:
            page = wiki.page(title)
            if not page.exists():
                failed.append(f"{title} (page not found)")
                continue
            cleaned = clean_text(page.text)
            if len(cleaned.split()) < 50:
                failed.append(f"{title} (too short)")
                continue
            for chunk in chunk_text(cleaned):
                KNOWLEDGE_BASE.append({"chunk": chunk, "source": page.title})
            loaded.append(title)
        except Exception as e:
            failed.append(f"{title} ({e})")

    status = f"✅ Loaded {len(loaded)} pages → {len(KNOWLEDGE_BASE)} chunks."
    if failed:
        status += f"  ⚠️ Skipped: {', '.join(failed)}"
    return status

# No auto-load here — user clicks the button to load
print("✅ Knowledge base functions ready.")

✅ Knowledge base functions ready.


In [21]:
# Cell 4 — LLM card generation
#
# SYSTEM PROMPT (clearly visible as required by the deliverable checklist)
# -----------------------------------------------------------------------
# This system prompt is intentionally strict:
#   • The model MUST only use facts present in the provided passage.
#   • It MUST NOT add background knowledge, make inferences, or hallucinate.
#   • The output is exactly 1–2 sentences, written for a general audience.
# -----------------------------------------------------------------------

SYSTEM_PROMPT = """You are a cybersecurity fact-card writer.
Your ONLY source of information is the Wikipedia passage the user supplies.
Rules you must follow without exception:
1. Write exactly 1 or 2 sentences — no more, no less.
2. Every factual claim must come directly from the supplied passage.
3. Do NOT add any information from your training data or general knowledge.
4. Do NOT repeat a fact that appeared in any previous card in this session.
5. Write in plain, engaging English suitable for a general audience.
6. Do NOT include headings, bullet points, or source citations in your output.
If the passage does not contain a useful standalone fact, output exactly: SKIP"""

def generate_card() -> str:
    if not KNOWLEDGE_BASE:
        return "⚠️ Knowledge base is empty. Please click LOAD TOPIC first."

    entry = random.choice(KNOWLEDGE_BASE)
    chunk, source = entry["chunk"], entry["source"]

    user_message = f"""Passage from Wikipedia ({source}):
\"\"\"
{chunk}
\"\"\"

Write your 1–2 sentence fact card now:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.9,
        max_tokens=120
    )

    insight = response.choices[0].message.content.strip()
    if insight.upper() == "SKIP":
        return generate_card()

    return f"{insight}\n\n— Source: {source}"

print("✅ Card generator ready.")

✅ Card generator ready.


In [24]:
TOPIC_PRESETS = {
    "🔐 Cybersecurity": [
        "Computer security", "Cybercrime", "Malware", "Phishing",
        "Ransomware", "Firewall (computing)", "Encryption",
        "Data breach", "Social engineering (security)",
        "Denial-of-service attack"
    ],
    "🤖 Artificial Intelligence": [
        "Artificial intelligence", "Machine learning", "Deep learning",
        "Natural language processing", "Neural network (machine learning)",
        "Reinforcement learning", "Computer vision",
        "Generative artificial intelligence", "Large language model",
        "Transformer (deep learning architecture)"
    ],
    "🌌 Space Exploration": [
        "Space exploration", "Mars", "International Space Station",
        "SpaceX", "Hubble Space Telescope", "Black hole",
        "Apollo program", "Exoplanet", "James Webb Space Telescope",
        "Voyager program"
    ]
}

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Exo+2:wght@300;600;800&display=swap');

body, .gradio-container {
    background: #0a0f1a !important;
    font-family: 'Exo 2', sans-serif !important;
}

/* Status box */
#status-box textarea,
#status-box input {
    background: #0d1f2d !important;
    border: 1px solid #00ff9d55 !important;
    color: #00ff9d !important;
    font-family: 'Share Tech Mono', monospace !important;
    font-size: 0.85rem !important;
    border-radius: 8px !important;
    padding: 10px 14px !important;
}

/* Fact card box */
#card-output textarea,
#card-output input {
    background: #0d1f2d !important;
    border: 2px solid #00c8ff88 !important;
    border-radius: 16px !important;
    color: #ffffff !important;
    font-family: 'Exo 2', sans-serif !important;
    font-size: 1.2rem !important;
    font-weight: 300 !important;
    line-height: 1.9 !important;
    padding: 24px 28px !important;
    min-height: 180px !important;
    box-shadow: 0 0 24px #00c8ff22 !important;
    resize: none !important;
}

/* Labels */
label, .block > label span {
    color: #00c8ff !important;
    font-family: 'Share Tech Mono', monospace !important;
    font-size: 0.8rem !important;
    letter-spacing: 2px !important;
    text-transform: uppercase !important;
}

/* Dropdown */
.wrap-inner,
select,
input[type="text"] {
    background: #f8fafc !important;
    color: #111827 !important;
    border: 1px solid #cbd5e1 !important;
}
.wrap {
    background: #ffffff !important;
    border: 1px solid #cbd5e1 !important;
}
.wrap span {
    color: #111827 !important;
}
ul.options {
    background: #ffffff !important;
    border: 1px solid #cbd5e1 !important;
}
ul.options li {
    color: #111827 !important;
}

ul.options li:hover {
    background: #e2f7ec !important;
    color: #111827 !important;
}

/* LOAD TOPIC button */
#load-btn button {
    background: #0d1f2d !important;
    color: #00ff9d !important;
    border: 1.5px solid #00ff9d !important;
    border-radius: 10px !important;
    font-family: 'Share Tech Mono', monospace !important;
    font-size: 0.9rem !important;
    letter-spacing: 1px !important;
    transition: background 0.2s !important;
}
#load-btn button:hover {
    background: #00ff9d22 !important;
}

/* NEXT CARD button */
#next-btn button {
    background: linear-gradient(90deg, #00ff9d, #00c8ff) !important;
    color: #050a0f !important;
    font-family: 'Share Tech Mono', monospace !important;
    font-size: 1.05rem !important;
    font-weight: 700 !important;
    letter-spacing: 3px !important;
    border: none !important;
    border-radius: 12px !important;
    padding: 16px 0 !important;
    width: 100% !important;
    cursor: pointer !important;
    transition: opacity 0.2s !important;
    text-transform: uppercase !important;
    margin-top: 8px !important;
}
#next-btn button:hover {
    opacity: 0.85 !important;
}

/* Hide Gradio branding */
footer { display: none !important; }
.built-with { display: none !important; }
div.svelte-1jfhh4o { display: none !important; }
"""

def handle_load_topic(topic_name):
    pages = TOPIC_PRESETS.get(topic_name, TOPIC_PRESETS["🔐 Cybersecurity"])
    return load_knowledge_base(pages)

def handle_next_card():
    return generate_card()

with gr.Blocks(css=CSS, title="CyberFeed", analytics_enabled=False) as demo:
    gr.HTML("""
    <div style="text-align:center; padding:32px 0 16px;">
      <div style="font-family:'Share Tech Mono',monospace; font-size:2.6rem;
                  color:#00ff9d; letter-spacing:8px; text-shadow:0 0 24px #00ff9d99;">
        ⬡ CYBERFEED
      </div>
      <div style="color:#00c8ffcc; font-family:'Exo 2',sans-serif;
                  font-size:0.9rem; margin-top:8px; letter-spacing:3px;">
        LIVE &nbsp;·&nbsp; AI-GENERATED &nbsp;·&nbsp; WIKIPEDIA-GROUNDED
      </div>
    </div>
    """)

    with gr.Row():
        topic_dd = gr.Dropdown(
            choices=list(TOPIC_PRESETS.keys()),
            value="🔐 Cybersecurity",
            label="Select Topic",
            scale=4
        )
        load_btn = gr.Button("⟳  LOAD TOPIC", elem_id="load-btn", scale=1)

    status_box = gr.Textbox(
        value="← Select a topic and click LOAD TOPIC to begin.",
        label="Status",
        interactive=False,
        elem_id="status-box",
        lines=1
    )

    card_out = gr.Textbox(
        value="Click  ▶ NEXT CARD  after loading a topic to see your first fact.",
        label="Fact Card",
        interactive=False,
        elem_id="card-output",
        lines=6
    )

    next_btn = gr.Button("▶  NEXT CARD", elem_id="next-btn")

    load_btn.click(fn=handle_load_topic, inputs=topic_dd, outputs=status_box)
    next_btn.click(fn=handle_next_card,  inputs=None,     outputs=card_out)

print("✅ Gradio interface built.")

✅ Gradio interface built.


In [28]:
# Cell 6 — Launch the app
demo.launch(share=True, show_api=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://af3a38156d06fb985c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:
demo.close()

Closing server running on port: 7864


---
## 📝 Reflection

The biggest challenge was preventing hallucinations while keeping the cards interesting. My first attempt used a vague prompt asking the model to "write an insight about cybersecurity," which led it to draw on its own training data rather than the supplied passage — classic hallucination. I solved this by writing a strict system prompt that (a) names the passage as the *only* permitted source, (b) explicitly forbids adding background knowledge, and (c) instructs the model to output the literal string `SKIP` if the chunk yields nothing useful, allowing the app to retry silently. A second challenge was Wikipedia's disambiguation pages crashing the fetch loop; wrapping each `wikipedia.page()` call in a `try/except` and logging failures without stopping the entire load fixed this gracefully. I also found that 400-word chunks were too long — the model would pick facts from the end of the chunk and ignore the beginning — so reducing chunks to 350 words improved coverage and variety noticeably. Finally, clearing `KNOWLEDGE_BASE = []` before every topic load was easy to forget but critical: without it, old chunks from a previous topic would mix with new ones, confusing the LLM. Adding a comment in the code and a visible status box in the UI made this transparent to the user.
